# Phase 5 — Inventory Intelligence

## Step 1— Input Validation

### Objective

Validate that the governed outputs from Phase 2 and Phase 3 contain the
required information for building:

1. Reorder Recommendation Engine
2. Slow-Mover Action Engine
3. Explainable inventory decision rules

The Phase 5 engines will use governed analytical outputs rather than
re-reading or recomputing logic from the original raw Excel files.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Phase 5 environment ready.")

Phase 5 environment ready.


In [2]:
from pathlib import Path

processed_path = Path("../data/processed")

files = sorted(processed_path.glob("*"))

print("Available processed files:\n")

for file in files:
    print(file.name)

Available processed files:



In [8]:
PROJECT_ROOT = Path.cwd().parents[1]

print("Project root:")
print(PROJECT_ROOT)

csv_files = sorted(PROJECT_ROOT.rglob("*.csv"))
excel_files = sorted(PROJECT_ROOT.rglob("*.xlsx"))

print(f"CSV files found: {len(csv_files)}")
print("=" * 70)

for file in csv_files:
    print(file.relative_to(PROJECT_ROOT))

print(f"\nExcel files found: {len(excel_files)}")
print("=" * 70)

for file in excel_files:
    print(file.relative_to(PROJECT_ROOT))

Project root:
d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System
CSV files found: 5
data\processed\dim_product.csv
data\processed\dim_store.csv
data\processed\fact_sales.csv
data\processed\fact_soa.csv
data\processed\fact_stock.csv

Excel files found: 1
docs\Phase_1_Define\03_Requirements_Acceptance_Criteria.xlsx


In [9]:
## Load all five tables
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

dim_product = pd.read_csv(DATA_PROCESSED / "dim_product.csv")
dim_store   = pd.read_csv(DATA_PROCESSED / "dim_store.csv")
fact_sales  = pd.read_csv(DATA_PROCESSED / "fact_sales.csv")
fact_stock  = pd.read_csv(DATA_PROCESSED / "fact_stock.csv")
fact_soa    = pd.read_csv(DATA_PROCESSED / "fact_soa.csv")

print("Governed tables loaded successfully.")

Governed tables loaded successfully.


In [10]:
## Validate shapes
tables = {
    "dim_product": dim_product,
    "dim_store": dim_store,
    "fact_sales": fact_sales,
    "fact_stock": fact_stock,
    "fact_soa": fact_soa
}

print("GOVERNED TABLE SHAPES")
print("=" * 60)

for name, df in tables.items():
    print(f"{name:<15} Rows: {len(df):>6} | Columns: {df.shape[1]:>2}")

GOVERNED TABLE SHAPES
dim_product     Rows:   1436 | Columns: 18
dim_store       Rows:      7 | Columns:  2
fact_sales      Rows:    966 | Columns: 18
fact_stock      Rows:   2681 | Columns:  9
fact_soa        Rows:    412 | Columns: 11


In [11]:
## Inspect columns & Sample records
for name, df in tables.items():

    print(f"\n{name.upper()}")
    print("=" * 70)

    for col in df.columns:
        print(col)

for name, df in tables.items():

    print(f"\n{name.upper()}")
    print("=" * 100)

    display(df.head())


DIM_PRODUCT
Product_ID
Product_Key
Product_Description
Product_Category
Record_Type
Source_Status
Source_Count
In_Sales
In_Stock
In_SOA
Sales_Stock_Code
Sales_Description
Sales_Category
Stock_Model
Stock_Description
Stock_Category
SOA_Model
SOA_Description

DIM_STORE
store_id
Store

FACT_SALES
Sales_Record_ID
Product_ID
Product_Key
Source_Month
Category
Stock Code
Description
Record_Type
Level
Sold Period
Transaction_Status
Unit Cost
Unit Price
Cost Sales
Sales Value
Profit
Profit %
Cost_Sales_Reconciliation_Flag

FACT_STOCK
Product_ID
Model
Store
Category
Description
Quantity
Stock_Status
Outstanding_Order_Qty
store_id

FACT_SOA
SOA_Record_ID
Product_ID
Product_Key
Model
Description
Starts
Ends
Window_Days
SOA
Original_Ends
Date_Correction_Flag

DIM_PRODUCT


,Product_ID,Product_Key,Product_Description,Product_Category,Record_Type,Source_Status,Source_Count,In_Sales,In_Stock,In_SOA,Sales_Stock_Code,Sales_Description,Sales_Category,Stock_Model,Stock_Description,Stock_Category,SOA_Model,SOA_Description
0,1,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02384-10,Garmin Lily Cream Gold & White,FITNESS,NaN,NaN,NaN,NaN,NaN
1,2,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-00,Garmin Venu 3 Smartwatch - Silver,FITNESS,NaN,NaN,NaN,NaN,NaN
2,3,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02784-01,Garmin Venu 3 Smartwatch - Slate,FITNESS,NaN,NaN,NaN,NaN,NaN
3,4,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,PRODUCT,SALES_ONLY,1,True,False,False,010-02839-00,"Garmin Lily 2, Cream Gold w/",FITNESS,NaN,NaN,NaN,NaN,NaN
4,5,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,PRODUCT,SALES_ONLY,1,True,False,False,01950,NUTRIBULLET PRO 4pc Starter Kit,BLENDERS,NaN,NaN,NaN,NaN,NaN



DIM_STORE


,store_id,Store
0,1,Belfast
1,2,Blanch
2,3,Cavan
3,4,Dundrum
4,5,Gorey



FACT_SALES


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH



FACT_STOCK


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty,store_id
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0,1
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,2
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,3
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0,4
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0,5



FACT_SOA


,SOA_Record_ID,Product_ID,Product_Key,Model,Description,Starts,Ends,Window_Days,SOA,Original_Ends,Date_Correction_Flag
0,1,11,107833-01,107833-01,Dyson Supersonic Hair Dryer,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
1,2,36,161818-01,161818-01,Dyson Supersonic Ceramic,2025-12-31,2026-02-03,35,66.50,2026-02-03,False
2,3,39,19750,19750,Russell Hobbs Rice Cooker 1.8Ltr,2026-02-01,2026-02-28,28,5.00,2026-02-28,False
3,4,48,21270,21270,Russell Hobbs White Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False
4,5,49,21271,21271,Russell Hobbs Black Textures Jug,2026-02-01,2026-02-28,28,2.29,2026-02-28,False


In [12]:
print("FACT_SALES COLUMNS")
print("=" * 70)

for col in fact_sales.columns:
    print(col)

display(fact_sales.head())

FACT_SALES COLUMNS
Sales_Record_ID
Product_ID
Product_Key
Source_Month
Category
Stock Code
Description
Record_Type
Level
Sold Period
Transaction_Status
Unit Cost
Unit Price
Cost Sales
Sales Value
Profit
Profit %
Cost_Sales_Reconciliation_Flag


,Sales_Record_ID,Product_ID,Product_Key,Source_Month,Category,Stock Code,Description,Record_Type,Level,Sold Period,Transaction_Status,Unit Cost,Unit Price,Cost Sales,Sales Value,Profit,Profit %,Cost_Sales_Reconciliation_Flag
0,1,1223,TLS169BOXE,Nov,ACCESSORIES,TLS169BOXE,Boxed Uni Floor Tool 30-38MM,PRODUCT,0,1,POSITIVE_SALES_ACTIVITY,9.31,29.99,9.31,24.99,15.68,62.75,MATCH
1,2,17,112.204,Nov,ACCESSORIES,112.204,TV Arial Lead 4.0m,PRODUCT,3,1,POSITIVE_SALES_ACTIVITY,1.28,2.99,1.28,2.49,1.21,48.59,MATCH
2,3,385,DLSC500,Nov,ACCESSORIES,DLSC500,Delonghi Descaler,PRODUCT,5,1,POSITIVE_SALES_ACTIVITY,7.77,14.49,7.77,7.50,-0.27,-3.60,MATCH
3,4,246,AF01,Nov,ACCESSORIES,AF01,VACUUM FRESHENERS AF101,PRODUCT,3,2,POSITIVE_SALES_ACTIVITY,2.30,3.49,4.60,3.32,-1.28,-38.55,MATCH
4,5,1076,SES007NEU0,Nov,ACCESSORIES,SES007NEU0,Sage Descaler (pack of 4),PRODUCT,6,2,POSITIVE_SALES_ACTIVITY,9.39,14.49,18.78,23.32,4.54,19.47,MATCH


In [13]:
print("FACT_STOCK COLUMNS")
print("=" * 70)

for col in fact_stock.columns:
    print(col)

display(fact_stock.head())

FACT_STOCK COLUMNS
Product_ID
Model
Store
Category
Description
Quantity
Stock_Status
Outstanding_Order_Qty
store_id


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty,store_id
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0,1
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,2
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,3
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0,4
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0,5


In [14]:
print("KEY VALIDATION")
print("=" * 70)

print("dim_product unique Product_ID:",
      dim_product["Product_ID"].is_unique)

print("fact_sales missing Product_ID:",
      fact_sales["Product_ID"].isna().sum()
      if "Product_ID" in fact_sales.columns else "Product_ID not present")

print("fact_stock missing Product_ID:",
      fact_stock["Product_ID"].isna().sum()
      if "Product_ID" in fact_stock.columns else "Product_ID not present")

KEY VALIDATION
dim_product unique Product_ID: True
fact_sales missing Product_ID: 0
fact_stock missing Product_ID: 0


### Findings

- `dim_product.Product_ID` is unique and provides the canonical product key.
- `fact_sales` contains no missing `Product_ID` values.
- `fact_stock` contains no missing `Product_ID` values.
- Store-level inventory is available at Product × Store grain.
- Sales demand can be derived from `Sold Period`.
- Current inventory is represented by `Quantity`.
- Outstanding purchase orders are available through `Outstanding_Order_Qty`.
- Raw Excel files are not required for Phase 5.

### Phase 5 Decision Grain

Inventory recommendations will be generated at:

**Product_ID × Store**

This allows the same product to receive different inventory decisions
depending on stock availability at each store.

### Step 2 — Build the Inventory Decision Base

In [15]:
sales_check_cols = [
    "Sold Period",
    "Sales Value",
    "Profit"
]

stock_check_cols = [
    "Quantity",
    "Outstanding_Order_Qty"
]

print("FACT_SALES DTYPES")
print("=" * 60)
print(fact_sales[sales_check_cols].dtypes)

print("\nFACT_STOCK DTYPES")
print("=" * 60)
print(fact_stock[stock_check_cols].dtypes)

FACT_SALES DTYPES
Sold Period      int64
Sales Value    float64
Profit         float64
dtype: object

FACT_STOCK DTYPES
Quantity                 int64
Outstanding_Order_Qty    int64
dtype: object


In [16]:
print("\nSALES SUMMARY")
print("=" * 60)

display(
    fact_sales[
        ["Sold Period", "Sales Value", "Profit"]
    ].describe()
)

print("\nSTOCK SUMMARY")
print("=" * 60)

display(
    fact_stock[
        ["Quantity", "Outstanding_Order_Qty"]
    ].describe()
)


SALES SUMMARY


,Sold Period,Sales Value,Profit
count,966.000000,966.000000,966.000000
mean,1.219462,289.183064,5.878934
std,0.894123,454.541089,177.314744
min,-5.000000,-4104.190000,-4104.230000
25%,1.000000,33.330000,-5.000000
50%,1.000000,138.750000,6.510000
75%,1.000000,399.792500,27.842500
max,12.000000,4000.000000,815.900000



STOCK SUMMARY


,Quantity,Outstanding_Order_Qty
count,2681.000000,2681.000000
mean,2.238344,0.004849
std,6.942050,0.074656
min,-2.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,0.000000
75%,2.000000,0.000000
max,145.000000,2.000000


In [17]:
print("SOURCE MONTHS")
print("=" * 60)

print(
    fact_sales["Source_Month"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nUnique source months:",
      fact_sales["Source_Month"].nunique())

SOURCE MONTHS
Source_Month
Dec    323
Jan    262
Nov    381
Name: count, dtype: int64

Unique source months: 3


In [18]:
product_demand = (
    fact_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Total_Units_Sold=("Sold Period", "sum"),
        Average_Sales_Velocity=("Sold Period", "mean"),
        Total_Sales_Value=("Sales Value", "sum"),
        Total_Profit=("Profit", "sum"),
        Sales_Periods=("Source_Month", "nunique")
    )
)

print("Product demand table created.")
print("Shape:", product_demand.shape)

display(product_demand.head())

Product demand table created.
Shape: (767, 6)


,Product_ID,Total_Units_Sold,Average_Sales_Velocity,Total_Sales_Value,Total_Profit,Sales_Periods
0,1,1,1.0,165.83,41.86,1
1,2,1,1.0,307.50,38.02,1
2,3,1,1.0,290.83,21.32,1
3,4,1,1.0,165.83,14.91,1
4,5,1,1.0,57.50,11.60,1


### Step 3 — Recreate the governed ABC classification

In [19]:
abc_analysis = product_demand.copy()

abc_analysis = abc_analysis.sort_values(
    "Total_Sales_Value",
    ascending=False
).reset_index(drop=True)

total_sales = abc_analysis["Total_Sales_Value"].sum()

abc_analysis["Sales_Contribution"] = (
    abc_analysis["Total_Sales_Value"] / total_sales
)

abc_analysis["Cumulative_Sales_Share"] = (
    abc_analysis["Sales_Contribution"].cumsum()
)

def assign_abc(cumulative_share):
    if cumulative_share <= 0.80:
        return "A"
    elif cumulative_share <= 0.95:
        return "B"
    else:
        return "C"

abc_analysis["ABC_Class"] = (
    abc_analysis["Cumulative_Sales_Share"]
    .apply(assign_abc)
)

display(
    abc_analysis[
        [
            "Product_ID",
            "Total_Sales_Value",
            "Cumulative_Sales_Share",
            "ABC_Class"
        ]
    ].head(20)
)

,Product_ID,Total_Sales_Value,Cumulative_Sales_Share,ABC_Class
0,1167,6818.30,0.024408,A
1,116,4000.00,0.038727,A
2,870,3165.00,0.050056,A
3,1320,2832.49,0.060196,A
4,1213,2764.17,0.070091,A
5,560,2683.34,0.079697,A
6,1425,2666.67,0.089243,A
7,856,2581.67,0.098484,A
8,1037,2580.83,0.107723,A
9,1246,2538.33,0.116809,A


In [20]:
## Check ABC distribution
abc_summary = (
    abc_analysis
    .groupby("ABC_Class")
    .agg(
        Products=("Product_ID", "count"),
        Sales_Value=("Total_Sales_Value", "sum")
    )
    .reset_index()
)

abc_summary["Sales_Share_%"] = (
    abc_summary["Sales_Value"]
    / abc_summary["Sales_Value"].sum()
    * 100
)

display(abc_summary)

,ABC_Class,Products,Sales_Value,Sales_Share_%
0,A,227,223145.64,79.880068
1,B,168,42205.51,15.108424
2,C,372,13999.69,5.011508


##### Demand & Stock Rule-Boundary Validation

Before defining inventory decision thresholds, the observed distributions of
sales velocity and stock are validated.

Special attention is given to:

- negative sales quantities,
- negative stock quantities,
- products appearing in different numbers of sales periods,
- skewed sales-velocity distributions,
- existing outstanding purchase orders.

Thresholds will be derived from the governed data rather than selected
arbitrarily.

In [21]:
negative_sales = fact_sales[
    fact_sales["Sold Period"] < 0
].copy()

print("NEGATIVE SOLD PERIOD RECORDS")
print("=" * 70)

print("Count:", len(negative_sales))

if len(negative_sales) > 0:
    display(
        negative_sales[
            [
                "Product_ID",
                "Source_Month",
                "Stock Code",
                "Description",
                "Sold Period",
                "Transaction_Status",
                "Sales Value",
                "Profit"
            ]
        ]
    )

NEGATIVE SOLD PERIOD RECORDS
Count: 15


,Product_ID,Source_Month,Stock Code,Description,Sold Period,Transaction_Status,Sales Value,Profit
48,616,Nov,INTEGRATA6,Elica 5414601 60cm Integrata,-1,NEGATIVE_SALES_ACTIVITY,-74.17,-13.82
221,1377,Nov,WM2621,OFA WALL MOUNT TILT FOR,-1,NEGATIVE_SALES_ACTIVITY,-24.99,-13.94
280,89,Nov,26771,Russell Hobbs Bronte Toaster Stone,-5,NEGATIVE_SALES_ACTIVITY,-249.95,-62.50
360,1339,Nov,WGG254Z0GB*Bosch,Series 6 10kg 1400 Spin,-1,NEGATIVE_SALES_ACTIVITY,-440.83,-42.18
416,424,Dec,EC9155.MB,DeLonghi La Specailista Arte Coffee,-1,NEGATIVE_SALES_ACTIVITY,-274.17,-21.50
424,380,Dec,DGE5861HM,AEG 80cm Canopy Cooker Hood,-1,NEGATIVE_SALES_ACTIVITY,-340.83,18.72
492,181,Dec,561727-01,Dyson Supersonic Nural Strawberry,-1,NEGATIVE_SALES_ACTIVITY,-332.50,-62.39
502,1366,Dec,WHULT900NBSONY,"Black Bluetooth 5.2, NC",-1,NEGATIVE_SALES_ACTIVITY,-99.17,14.34
558,1117,Dec,SKE735BTR4,Sage Black Truffle Kettle,-1,NEGATIVE_SALES_ACTIVITY,-79.17,-11.80
585,1166,Dec,T2080GA1,Eufy Robot Vacuum S1 Pro,-1,NEGATIVE_SALES_ACTIVITY,-750.00,-28.50


In [22]:
transaction_summary = (
    fact_sales
    .groupby("Transaction_Status", dropna=False)
    .agg(
        Records=("Sales_Record_ID", "count"),
        Units=("Sold Period", "sum"),
        Sales_Value=("Sales Value", "sum")
    )
    .reset_index()
)

display(transaction_summary)

,Transaction_Status,Records,Units,Sales_Value
0,NEGATIVE_SALES_ACTIVITY,15,-19,-3548.22
1,POSITIVE_SALES_ACTIVITY,941,1197,282951.59
2,ZERO_UNIT_FINANCIAL_ADJUSTMENT,10,0,-52.53


In [23]:
display(
    fact_sales.groupby(
        ["Transaction_Status", "Source_Month"],
        dropna=False
    )["Sold Period"]
    .agg(["count", "sum", "min", "max"])
)

count  sum  min  max
Transaction_Status             Source_Month                      
NEGATIVE_SALES_ACTIVITY        Dec               7   -7   -1   -1
                               Jan               4   -4   -1   -1
                               Nov               4   -8   -5   -1
POSITIVE_SALES_ACTIVITY        Dec             313  401    1   12
                               Jan             255  326    1    8
                               Nov             373  470    1    9
ZERO_UNIT_FINANCIAL_ADJUSTMENT Dec               3    0    0    0
                               Jan               3    0    0    0
                               Nov               4    0    0    0

In [24]:
period_coverage = (
    product_demand["Sales_Periods"]
    .value_counts()
    .sort_index()
    .rename_axis("Sales_Periods")
    .reset_index(name="Products")
)

display(period_coverage)

,Sales_Periods,Products
0,1,602
1,2,138
2,3,27


In [25]:
## Examine sales-velocity distribution
velocity_summary = (
    product_demand["Average_Sales_Velocity"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

display(velocity_summary)

count    767.000000
mean       1.162277
std        0.635028
min       -5.000000
10%        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
90%        2.000000
95%        2.000000
max        6.000000
Name: Average_Sales_Velocity, dtype: float64

In [26]:
velocity_frequency = (
    product_demand["Average_Sales_Velocity"]
    .round(2)
    .value_counts()
    .sort_index()
    .reset_index()
)

velocity_frequency.columns = [
    "Average_Sales_Velocity",
    "Products"
]

display(velocity_frequency.head(30))

,Average_Sales_Velocity,Products
0,-5.00,1
1,-1.00,5
2,0.00,9
3,0.50,5
4,1.00,615
5,1.33,4
6,1.50,31
7,1.67,5
8,2.00,61
9,2.33,3


In [27]:
negative_stock = fact_stock[
    fact_stock["Quantity"] < 0
].copy()

print("NEGATIVE STOCK RECORDS")
print("=" * 70)

print("Count:", len(negative_stock))

if len(negative_stock) > 0:
    display(
        negative_stock[
            [
                "Product_ID",
                "Model",
                "Store",
                "Category",
                "Description",
                "Quantity",
                "Stock_Status",
                "Outstanding_Order_Qty"
            ]
        ]
    )

NEGATIVE STOCK RECORDS
Count: 12


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty
18,223,980533,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT BLACK MANUAL,-2,CUSTOMER_ORDER_OUTSTANDING,2
102,272,B54CR71G0B,Gorey,SINGLE OVENS,Neff N70 Graphite Single Oven,-1,CUSTOMER_ORDER_OUTSTANDING,1
334,394,DV90DB8845GBU1,Navan,TUMBLE DRYERS,Samsung Series 8 9kg Heat Pump Dryer,-1,CUSTOMER_ORDER_OUTSTANDING,1
698,540,GUN21VFE0G,Navan,U/C FREEZER,Bosch U/C Integrated Freezer Flat Hinge,-1,CUSTOMER_ORDER_OUTSTANDING,1
855,610,IMA764MYTIMEUK,Blanch,WASHING MACHINES,Indesit 7kg 1400 Spin Washing Machine,-1,CUSTOMER_ORDER_OUTSTANDING,1
1541,897,P154MDTC,Blanch,HOBS,Powerpoint Touch Control Ceramic Hob,-1,CUSTOMER_ORDER_OUTSTANDING,1
1616,911,P24510M6WH,Sandyford,DISHWASHERS,Powerpoint 45cm White Dishwasher,-1,CUSTOMER_ORDER_OUTSTANDING,1
1709,928,P455LM3W,Blanch,U/C FRIDGE,Powerpoint 55cm U/C Larder Fridge,-1,CUSTOMER_ORDER_OUTSTANDING,1
1804,946,PKE611CA3E,Navan,HOBS,*Bosch 60cm Serie 2 Ceramic Hob with Knobs,-1,CUSTOMER_ORDER_OUTSTANDING,1
1993,1051,S187ZCX03G,Navan,INT DISHWASHERS,Neff N70 Integrated Dishwasher,-1,CUSTOMER_ORDER_OUTSTANDING,1


In [28]:
outstanding_summary = (
    fact_stock["Outstanding_Order_Qty"]
    .value_counts()
    .sort_index()
    .rename_axis("Outstanding_Order_Qty")
    .reset_index(name="Records")
)

display(outstanding_summary)

,Outstanding_Order_Qty,Records
0,0,2669
1,1,11
2,2,1


In [29]:
stock_with_orders = fact_stock[
    fact_stock["Outstanding_Order_Qty"] > 0
]

print(
    "Store-product records with outstanding orders:",
    len(stock_with_orders)
)

display(
    stock_with_orders[
        [
            "Product_ID",
            "Model",
            "Store",
            "Quantity",
            "Outstanding_Order_Qty",
            "Stock_Status"
        ]
    ].head(20)
)

Store-product records with outstanding orders: 12


,Product_ID,Model,Store,Quantity,Outstanding_Order_Qty,Stock_Status
18,223,980533,Gorey,-2,2,CUSTOMER_ORDER_OUTSTANDING
102,272,B54CR71G0B,Gorey,-1,1,CUSTOMER_ORDER_OUTSTANDING
334,394,DV90DB8845GBU1,Navan,-1,1,CUSTOMER_ORDER_OUTSTANDING
698,540,GUN21VFE0G,Navan,-1,1,CUSTOMER_ORDER_OUTSTANDING
855,610,IMA764MYTIMEUK,Blanch,-1,1,CUSTOMER_ORDER_OUTSTANDING
1541,897,P154MDTC,Blanch,-1,1,CUSTOMER_ORDER_OUTSTANDING
1616,911,P24510M6WH,Sandyford,-1,1,CUSTOMER_ORDER_OUTSTANDING
1709,928,P455LM3W,Blanch,-1,1,CUSTOMER_ORDER_OUTSTANDING
1804,946,PKE611CA3E,Navan,-1,1,CUSTOMER_ORDER_OUTSTANDING
1993,1051,S187ZCX03G,Navan,-1,1,CUSTOMER_ORDER_OUTSTANDING


#####  Demand Treatment Decision

For inventory replenishment purposes:

- `POSITIVE_SALES_ACTIVITY` represents observed customer demand.
- `NEGATIVE_SALES_ACTIVITY` is retained for financial analysis but excluded
  from replenishment demand.
- `ZERO_UNIT_FINANCIAL_ADJUSTMENT` is excluded from unit-demand calculations.
- Negative stock is preserved as an operational condition.
- Existing outstanding orders are incorporated into inventory decisions to
  prevent duplicate reorder recommendations.

Because historical coverage is sparse and uneven, Phase 5 uses observed
positive sales activity as a rule-based demand signal rather than treating
it as a statistical forecast.

In [30]:
## Build clean positive-demand table
positive_sales = fact_sales[
    fact_sales["Transaction_Status"] == "POSITIVE_SALES_ACTIVITY"
].copy()

print("Positive sales records:", len(positive_sales))
print("Positive units sold:", positive_sales["Sold Period"].sum())

Positive sales records: 941
Positive units sold: 1197


In [31]:
product_demand_clean = (
    positive_sales
    .groupby("Product_ID", as_index=False)
    .agg(
        Total_Positive_Units=("Sold Period", "sum"),
        Avg_Positive_Velocity=("Sold Period", "mean"),
        Max_Monthly_Units=("Sold Period", "max"),
        Total_Sales_Value=("Sales Value", "sum"),
        Total_Profit=("Profit", "sum"),
        Active_Sales_Periods=("Source_Month", "nunique")
    )
)

print("Clean demand table shape:", product_demand_clean.shape)

display(product_demand_clean.head())

Clean demand table shape: (759, 7)


,Product_ID,Total_Positive_Units,Avg_Positive_Velocity,Max_Monthly_Units,Total_Sales_Value,Total_Profit,Active_Sales_Periods
0,1,1,1.0,1,165.83,41.86,1
1,2,1,1.0,1,307.50,38.02,1
2,3,1,1.0,1,290.83,21.32,1
3,4,1,1.0,1,165.83,14.91,1
4,5,1,1.0,1,57.50,11.60,1


### Step 4 — Define Sales Velocity Bands

In [32]:
## Assign velocity bands
def assign_velocity_band(v):
    if v <= 1:
        return "LOW"
    elif v <= 2:
        return "MEDIUM"
    else:
        return "HIGH"


product_demand_clean["Velocity_Band"] = (
    product_demand_clean["Avg_Positive_Velocity"]
    .apply(assign_velocity_band)
)

velocity_band_summary = (
    product_demand_clean["Velocity_Band"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
    .rename_axis("Velocity_Band")
    .reset_index(name="Products")
)

display(velocity_band_summary)

,Velocity_Band,Products
0,LOW,624
1,MEDIUM,102
2,HIGH,33


In [33]:
## Recalculate ABC on valid sales activity
abc_clean = (
    product_demand_clean[
        ["Product_ID", "Total_Sales_Value"]
    ]
    .sort_values("Total_Sales_Value", ascending=False)
    .reset_index(drop=True)
)

total_positive_sales_value = abc_clean["Total_Sales_Value"].sum()

abc_clean["Sales_Contribution"] = (
    abc_clean["Total_Sales_Value"]
    / total_positive_sales_value
)

abc_clean["Cumulative_Sales_Share"] = (
    abc_clean["Sales_Contribution"].cumsum()
)


def assign_abc(cumulative_share):
    if cumulative_share <= 0.80:
        return "A"
    elif cumulative_share <= 0.95:
        return "B"
    else:
        return "C"


abc_clean["ABC_Class"] = (
    abc_clean["Cumulative_Sales_Share"]
    .apply(assign_abc)
)

abc_clean_summary = (
    abc_clean
    .groupby("ABC_Class")
    .agg(
        Products=("Product_ID", "count"),
        Sales_Value=("Total_Sales_Value", "sum")
    )
    .reset_index()
)

abc_clean_summary["Sales_Share_%"] = (
    abc_clean_summary["Sales_Value"]
    / abc_clean_summary["Sales_Value"].sum()
    * 100
)

display(abc_clean_summary)

,ABC_Class,Products,Sales_Value,Sales_Share_%
0,A,234,226245.62,79.959127
1,B,176,42522.59,15.028221
2,C,349,14183.38,5.012653


In [34]:
## Build the Inventory Intelligence Base
inventory_base = (
    fact_stock
    .merge(
        product_demand_clean[
            [
                "Product_ID",
                "Total_Positive_Units",
                "Avg_Positive_Velocity",
                "Max_Monthly_Units",
                "Active_Sales_Periods",
                "Velocity_Band"
            ]
        ],
        on="Product_ID",
        how="left"
    )
    .merge(
        abc_clean[
            [
                "Product_ID",
                "ABC_Class"
            ]
        ],
        on="Product_ID",
        how="left"
    )
)

print("Inventory Intelligence Base")
print("=" * 60)
print("Shape:", inventory_base.shape)

display(inventory_base.head())

Inventory Intelligence Base
Shape: (2681, 15)


,Product_ID,Model,Store,Category,Description,Quantity,Stock_Status,Outstanding_Order_Qty,store_id,Total_Positive_Units,Avg_Positive_Velocity,Max_Monthly_Units,Active_Sales_Periods,Velocity_Band,ABC_Class
0,221,980531,Belfast,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,2,IN_STOCK,0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,221,980531,Blanch,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,2,NaN,NaN,NaN,NaN,NaN,NaN
2,221,980531,Cavan,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,3,IN_STOCK,0,3,NaN,NaN,NaN,NaN,NaN,NaN
3,221,980531,Dundrum,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,1,IN_STOCK,0,4,NaN,NaN,NaN,NaN,NaN,NaN
4,221,980531,Gorey,MICROWAVE OVENS,DIMPLEX 20 LITRE 800 WATT WHITE MANUAL,0,OUT_OF_STOCK,0,5,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
## Handle no-sales products explicitly
inventory_base["Demand_Status"] = np.where(
    inventory_base["Avg_Positive_Velocity"].isna(),
    "NO_OBSERVED_SALES",
    "OBSERVED_SALES"
)

inventory_base["Velocity_Band"] = (
    inventory_base["Velocity_Band"]
    .fillna("NO_SALES")
)

inventory_base["ABC_Class"] = (
    inventory_base["ABC_Class"]
    .fillna("UNCLASSIFIED")
)

inventory_base["Total_Positive_Units"] = (
    inventory_base["Total_Positive_Units"]
    .fillna(0)
)

inventory_base["Active_Sales_Periods"] = (
    inventory_base["Active_Sales_Periods"]
    .fillna(0)
    .astype(int)
)

print("Demand status:")
print(inventory_base["Demand_Status"].value_counts())

print("\nVelocity bands:")
print(inventory_base["Velocity_Band"].value_counts())

print("\nABC classes:")
print(inventory_base["ABC_Class"].value_counts())

Demand status:
Demand_Status
NO_OBSERVED_SALES    2289
OBSERVED_SALES        392
Name: count, dtype: int64

Velocity bands:
Velocity_Band
NO_SALES    2289
LOW          343
MEDIUM        42
HIGH           7
Name: count, dtype: int64

ABC classes:
ABC_Class
UNCLASSIFIED    2289
A                245
B                112
C                 35
Name: count, dtype: int64


In [36]:
## Validate grain
duplicate_grain = (
    inventory_base
    .duplicated(
        subset=["Product_ID", "Store"],
        keep=False
    )
    .sum()
)

print("Rows:", len(inventory_base))
print(
    "Unique Product × Store combinations:",
    inventory_base[
        ["Product_ID", "Store"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate Product × Store rows:",
    duplicate_grain
)

Rows: 2681
Unique Product × Store combinations: 2681
Duplicate Product × Store rows: 0


### Step 5 — Reorder Recommendation Engine

In [37]:
## Create operational stock bands
def assign_stock_band(qty):
    if qty < 0:
        return "NEGATIVE"
    elif qty == 0:
        return "OUT_OF_STOCK"
    elif qty <= 2:
        return "LOW"
    elif qty <= 5:
        return "ADEQUATE"
    else:
        return "HIGH"


inventory_base["Stock_Band"] = (
    inventory_base["Quantity"]
    .apply(assign_stock_band)
)

stock_band_summary = (
    inventory_base["Stock_Band"]
    .value_counts()
    .reindex([
        "NEGATIVE",
        "OUT_OF_STOCK",
        "LOW",
        "ADEQUATE",
        "HIGH"
    ])
    .rename_axis("Stock_Band")
    .reset_index(name="Store_Product_Records")
)

display(stock_band_summary)

,Stock_Band,Store_Product_Records
0,NEGATIVE,12
1,OUT_OF_STOCK,793
2,LOW,1271
3,ADEQUATE,415
4,HIGH,190


In [38]:
## Add ABC priority
abc_priority_map = {
    "A": 3,
    "B": 2,
    "C": 1,
    "UNCLASSIFIED": 0
}

inventory_base["ABC_Priority"] = (
    inventory_base["ABC_Class"]
    .map(abc_priority_map)
)

In [39]:
velocity_priority_map = {
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1,
    "NO_SALES": 0
}

inventory_base["Velocity_Priority"] = (
    inventory_base["Velocity_Band"]
    .map(velocity_priority_map)
)

In [40]:
## Define the recommendation rules
def reorder_recommendation(row):

    qty = row["Quantity"]
    outstanding = row["Outstanding_Order_Qty"]
    velocity = row["Velocity_Band"]
    abc = row["ABC_Class"]

    # ------------------------------------------------
    # 1. Customer order already outstanding
    # ------------------------------------------------
    if qty < 0 and outstanding > 0:
        return "MONITOR CUSTOMER ORDER"

    # ------------------------------------------------
    # 2. Negative stock without an outstanding order
    # ------------------------------------------------
    if qty < 0 and outstanding == 0:
        return "REORDER - CRITICAL"

    # ------------------------------------------------
    # 3. Out of stock
    # ------------------------------------------------
    if qty == 0:

        if velocity == "HIGH" or abc == "A":
            return "REORDER - HIGH PRIORITY"

        if velocity in ["MEDIUM", "LOW"]:
            return "REORDER"

        return "REVIEW"

    # ------------------------------------------------
    # 4. Low stock: 1–2 units
    # ------------------------------------------------
    if 1 <= qty <= 2:

        if velocity == "HIGH":
            return "REORDER - HIGH PRIORITY"

        if velocity == "MEDIUM" and abc in ["A", "B"]:
            return "REORDER"

        if abc == "A":
            return "REVIEW"

        return "NO ACTION"

    # ------------------------------------------------
    # 5. Stock >= 3
    # ------------------------------------------------
    return "NO ACTION"


inventory_base["Reorder_Recommendation"] = (
    inventory_base.apply(
        reorder_recommendation,
        axis=1
    )
)

In [41]:
## Add explainable reason codes
def recommendation_reason(row):

    qty = row["Quantity"]
    outstanding = row["Outstanding_Order_Qty"]
    velocity = row["Velocity_Band"]
    abc = row["ABC_Class"]

    if qty < 0 and outstanding > 0:
        return "Negative stock linked to an outstanding customer order"

    if qty < 0:
        return "Negative stock with no outstanding order recorded"

    if qty == 0 and (velocity == "HIGH" or abc == "A"):
        return "Out of stock with high demand or A-class priority"

    if qty == 0 and velocity in ["MEDIUM", "LOW"]:
        return "Out of stock with observed positive demand"

    if qty == 0:
        return "Out of stock but no observed positive sales signal"

    if qty <= 2 and velocity == "HIGH":
        return "Low stock with high sales velocity"

    if qty <= 2 and velocity == "MEDIUM" and abc in ["A", "B"]:
        return "Low stock with medium velocity and high ABC importance"

    if qty <= 2 and abc == "A":
        return "Low stock for an A-class product"

    return "Stock level does not currently breach reorder rules"


inventory_base["Recommendation_Reason"] = (
    inventory_base.apply(
        recommendation_reason,
        axis=1
    )
)

In [42]:
## Recommendation distribution
recommendation_summary = (
    inventory_base["Reorder_Recommendation"]
    .value_counts()
    .rename_axis("Recommendation")
    .reset_index(name="Store_Product_Records")
)

recommendation_summary["Share_%"] = (
    recommendation_summary["Store_Product_Records"]
    / len(inventory_base)
    * 100
).round(2)

display(recommendation_summary)

,Recommendation,Store_Product_Records,Share_%
0,NO ACTION,1753,65.39
1,REVIEW,806,30.06
2,REORDER - HIGH PRIORITY,59,2.20
3,REORDER,51,1.90
4,MONITOR CUSTOMER ORDER,12,0.45


In [44]:
## Inspect actual reorder candidates
reorder_candidates = inventory_base[
    inventory_base["Reorder_Recommendation"].isin([
        "REORDER - CRITICAL",
        "REORDER - HIGH PRIORITY",
        "REORDER"
    ])
].copy()

reorder_candidates = reorder_candidates.sort_values(
    ["ABC_Priority", "Velocity_Priority", "Quantity"],
    ascending=[False, False, True]
)

print("Total reorder candidates:", len(reorder_candidates))

display(
    reorder_candidates[
        [
            "Product_ID",
            "Model",
            "Store",
            "Category",
            "Quantity",
            "ABC_Class",
            "Velocity_Band",
            "Avg_Positive_Velocity",
            "Reorder_Recommendation",
            "Recommendation_Reason"
        ]
    ].head(30)
)

Total reorder candidates: 110


,Product_ID,Model,Store,Category,Quantity,ABC_Class,Velocity_Band,Avg_Positive_Velocity,Reorder_Recommendation,Recommendation_Reason
2472,1320,WEK365WCS,Blanch,WASHING MACHINES,0,A,HIGH,3.0,REORDER - HIGH PRIORITY,Out of stock with high demand or A-class priority
2476,1320,WEK365WCS,Navan,WASHING MACHINES,0,A,HIGH,3.0,REORDER - HIGH PRIORITY,Out of stock with high demand or A-class priority
2471,1320,WEK365WCS,Belfast,WASHING MACHINES,1,A,HIGH,3.0,REORDER - HIGH PRIORITY,Low stock with high sales velocity
2473,1320,WEK365WCS,Cavan,WASHING MACHINES,1,A,HIGH,3.0,REORDER - HIGH PRIORITY,Low stock with high sales velocity
2474,1320,WEK365WCS,Dundrum,WASHING MACHINES,1,A,HIGH,3.0,REORDER - HIGH PRIORITY,Low stock with high sales velocity
2477,1320,WEK365WCS,Sandyford,WASHING MACHINES,1,A,HIGH,3.0,REORDER - HIGH PRIORITY,Low stock with high sales velocity
405,439,EDHI618WD,Sandyford,TUMBLE DRYERS,0,A,MEDIUM,2.0,REORDER - HIGH PRIORITY,Out of stock with high demand or A-class priority
747,560,H7464BPBL,Navan,SINGLE OVENS,0,A,MEDIUM,2.0,REORDER - HIGH PRIORITY,Out of stock with high demand or A-class priority
2203,1213,TEH785WP,Navan,TUMBLE DRYERS,0,A,MEDIUM,1.5,REORDER - HIGH PRIORITY,Out of stock with high demand or A-class priority
318,381,DI362DQ,Dundrum,INT DISHWASHERS,1,A,MEDIUM,2.0,REORDER,Low stock with medium velocity and high ABC im...


In [45]:
## Critical sanity checks
print("REORDER ENGINE VALIDATION")
print("=" * 70)

# Negative stock + outstanding customer order
customer_order_check = inventory_base[
    (inventory_base["Quantity"] < 0) &
    (inventory_base["Outstanding_Order_Qty"] > 0)
]

print(
    "Negative stock + outstanding order:",
    len(customer_order_check)
)

print(
    "Correctly routed to MONITOR CUSTOMER ORDER:",
    (
        customer_order_check["Reorder_Recommendation"]
        == "MONITOR CUSTOMER ORDER"
    ).all()
)

# Duplicate grain
print(
    "\nDuplicate Product × Store:",
    inventory_base.duplicated(
        ["Product_ID", "Store"]
    ).sum()
)

# Missing recommendations
print(
    "Missing recommendations:",
    inventory_base["Reorder_Recommendation"].isna().sum()
)

REORDER ENGINE VALIDATION
Negative stock + outstanding order: 12
Correctly routed to MONITOR CUSTOMER ORDER: True

Duplicate Product × Store: 0
Missing recommendations: 0


##### Reorder Engine Validation Findings

The rule-based Reorder Recommendation Engine was successfully applied at
Product × Store grain across 2,681 inventory records.

#### Results

- 1,753 records (65.39%) → No Action
- 806 records (30.06%) → Review
- 59 records (2.20%) → Reorder — High Priority
- 51 records (1.90%) → Reorder
- 12 records (0.45%) → Monitor Customer Order

A total of 110 store-product records were identified as active replenishment
candidates.

All 12 negative-stock records were associated with outstanding customer
orders and were correctly routed to `MONITOR CUSTOMER ORDER`, preventing
duplicate replenishment recommendations.

Validation confirmed:

- zero duplicate Product × Store records,
- zero missing recommendations,
- correct handling of outstanding customer orders,
- ABC class influences replenishment priority rather than independently
  triggering an order,
- sales velocity and stock availability jointly determine replenishment
  urgency.

The engine is therefore suitable to proceed to slow-mover analysis and
subsequent rule-boundary testing.

### Step 6 — Slow-Mover Action Logic

In [46]:
## Create slow-mover rule
def slow_mover_action(row):

    stock_band = row["Stock_Band"]
    velocity = row["Velocity_Band"]

    # Strongest slow-mover signal
    if velocity == "NO_SALES" and stock_band == "HIGH":
        return "PROMOTION / MARKDOWN REVIEW"

    # Observed demand exists, but movement is weak
    if velocity == "LOW" and stock_band == "HIGH":
        return "SLOW MOVER REVIEW"

    # No observed sales with moderate inventory exposure
    if velocity == "NO_SALES" and stock_band == "ADEQUATE":
        return "MONITOR"

    return "NO ACTION"


inventory_base["Slow_Mover_Action"] = (
    inventory_base.apply(
        slow_mover_action,
        axis=1
    )
)

In [47]:
## Add reason codes
def slow_mover_reason(row):

    stock_band = row["Stock_Band"]
    velocity = row["Velocity_Band"]
    qty = row["Quantity"]

    if velocity == "NO_SALES" and stock_band == "HIGH":
        return (
            f"No observed positive sales with high stock ({qty} units)"
        )

    if velocity == "LOW" and stock_band == "HIGH":
        return (
            f"Low sales velocity with high stock ({qty} units)"
        )

    if velocity == "NO_SALES" and stock_band == "ADEQUATE":
        return (
            f"No observed positive sales with adequate stock ({qty} units)"
        )

    return "No current slow-mover rule breached"


inventory_base["Slow_Mover_Reason"] = (
    inventory_base.apply(
        slow_mover_reason,
        axis=1
    )
)

In [48]:
slow_mover_summary = (
    inventory_base["Slow_Mover_Action"]
    .value_counts()
    .rename_axis("Slow_Mover_Action")
    .reset_index(name="Store_Product_Records")
)

slow_mover_summary["Share_%"] = (
    slow_mover_summary["Store_Product_Records"]
    / len(inventory_base)
    * 100
).round(2)

display(slow_mover_summary)

,Slow_Mover_Action,Store_Product_Records,Share_%
0,NO ACTION,2150,80.19
1,MONITOR,351,13.09
2,PROMOTION / MARKDOWN REVIEW,155,5.78
3,SLOW MOVER REVIEW,25,0.93


In [49]:
## Inspect actionable slow movers
slow_mover_candidates = inventory_base[
    inventory_base["Slow_Mover_Action"].isin([
        "PROMOTION / MARKDOWN REVIEW",
        "SLOW MOVER REVIEW"
    ])
].copy()

slow_mover_candidates = slow_mover_candidates.sort_values(
    ["Quantity", "ABC_Priority"],
    ascending=[False, True]
)

print(
    "Actionable slow-mover candidates:",
    len(slow_mover_candidates)
)

display(
    slow_mover_candidates[
        [
            "Product_ID",
            "Model",
            "Store",
            "Category",
            "Quantity",
            "ABC_Class",
            "Velocity_Band",
            "Avg_Positive_Velocity",
            "Slow_Mover_Action",
            "Slow_Mover_Reason"
        ]
    ].head(30)
)

Actionable slow-mover candidates: 180


,Product_ID,Model,Store,Category,Quantity,ABC_Class,Velocity_Band,Avg_Positive_Velocity,Slow_Mover_Action,Slow_Mover_Reason
2664,1427,YC-MS02U-S,Gorey,MICROWAVE OVENS,145,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (14...
1943,1031,RIFF70306NM,Gorey,INT FRIDGE FREEZERS,133,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (13...
2573,1383,WMA1270WH,Gorey,WASHING MACHINES,126,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (12...
305,377,DF63,Gorey,INT DISHWASHERS,121,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (12...
2118,1150,SO107IX,Gorey,SINGLE OVENS,119,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (11...
2160,1207,TDHPB80WH,Gorey,TUMBLE DRYERS,93,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (93...
1453,829,NM31FSBL,Gorey,MICROWAVE OVENS,80,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (80...
543,461,ERACIXA60,Gorey,COOKER HOODS,71,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (71...
1495,836,NZ64F3NM1AB/UR,Gorey,HOBS,56,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (56...
858,610,IMA764MYTIMEUK,Gorey,WASHING MACHINES,51,UNCLASSIFIED,NO_SALES,NaN,PROMOTION / MARKDOWN REVIEW,No observed positive sales with high stock (51...


In [50]:
## Check for conflicting recommendations
reorder_actions = [
    "REORDER - CRITICAL",
    "REORDER - HIGH PRIORITY",
    "REORDER"
]

slow_actions = [
    "PROMOTION / MARKDOWN REVIEW",
    "SLOW MOVER REVIEW"
]

conflicts = inventory_base[
    inventory_base["Reorder_Recommendation"].isin(reorder_actions)
    &
    inventory_base["Slow_Mover_Action"].isin(slow_actions)
]

print("REORDER × SLOW-MOVER CONFLICT CHECK")
print("=" * 70)

print("Conflicting records:", len(conflicts))

if len(conflicts) > 0:
    display(
        conflicts[
            [
                "Product_ID",
                "Model",
                "Store",
                "Quantity",
                "Velocity_Band",
                "ABC_Class",
                "Reorder_Recommendation",
                "Slow_Mover_Action"
            ]
        ]
    )

REORDER × SLOW-MOVER CONFLICT CHECK
Conflicting records: 0


### Slow-Mover Engine Validation Findings
The slow-mover decision engine was successfully evaluated across all
2,681 Product × Store inventory records.

#### Results

- 2,150 records (80.19%) → No Action
- 351 records (13.09%) → Monitor
- 155 records (5.78%) → Promotion / Markdown Review
- 25 records (0.93%) → Slow Mover Review

A total of 180 store-product records were identified as actionable
slow-mover candidates.

The strongest inventory-exposure signal was:

**No matched positive sales activity + High stock**

These records are routed to management review rather than automatically
triggering markdowns because the available sales history covers only the
current Nov–Jan analytical window and is not store-level demand history.

Conflict testing identified zero records simultaneously classified as
replenishment and actionable slow-mover candidates.

The slow-mover engine therefore passes the current Phase 5 validation gate.

### Step 7 — Unified Inventory Decision Layer

In [51]:
def assign_inventory_decision(row):

    reorder = row["Reorder_Recommendation"]
    slow = row["Slow_Mover_Action"]

    # Highest operational urgency
    if reorder == "REORDER - CRITICAL":
        return "URGENT REPLENISHMENT"

    if reorder == "REORDER - HIGH PRIORITY":
        return "PRIORITY REPLENISHMENT"

    if reorder == "REORDER":
        return "REPLENISH"

    # Existing customer-order commitment
    if reorder == "MONITOR CUSTOMER ORDER":
        return "MONITOR CUSTOMER ORDER"

    # Excess / slow-moving inventory
    if slow == "PROMOTION / MARKDOWN REVIEW":
        return "PROMOTION REVIEW"

    if slow == "SLOW MOVER REVIEW":
        return "SLOW MOVER REVIEW"

    # Other inventory conditions requiring attention
    if reorder == "REVIEW":
        return "STOCK REVIEW"

    if slow == "MONITOR":
        return "MONITOR"

    return "NO ACTION"


inventory_base["Inventory_Decision"] = (
    inventory_base.apply(
        assign_inventory_decision,
        axis=1
    )
)

In [52]:
## Add decision priority
decision_priority = {
    "URGENT REPLENISHMENT": 1,
    "PRIORITY REPLENISHMENT": 2,
    "REPLENISH": 3,
    "MONITOR CUSTOMER ORDER": 4,
    "PROMOTION REVIEW": 5,
    "SLOW MOVER REVIEW": 6,
    "STOCK REVIEW": 7,
    "MONITOR": 8,
    "NO ACTION": 9
}

inventory_base["Decision_Priority"] = (
    inventory_base["Inventory_Decision"]
    .map(decision_priority)
)

In [53]:
## Create unified decision reason
def final_decision_reason(row):

    decision = row["Inventory_Decision"]

    if decision in [
        "URGENT REPLENISHMENT",
        "PRIORITY REPLENISHMENT",
        "REPLENISH",
        "MONITOR CUSTOMER ORDER",
        "STOCK REVIEW"
    ]:
        return row["Recommendation_Reason"]

    if decision in [
        "PROMOTION REVIEW",
        "SLOW MOVER REVIEW",
        "MONITOR"
    ]:
        return row["Slow_Mover_Reason"]

    return "No current inventory intervention required"


inventory_base["Decision_Reason"] = (
    inventory_base.apply(
        final_decision_reason,
        axis=1
    )
)

In [54]:
decision_summary = (
    inventory_base
    .groupby(
        ["Decision_Priority", "Inventory_Decision"],
        as_index=False
    )
    .agg(
        Store_Product_Records=("Product_ID", "count")
    )
    .sort_values("Decision_Priority")
)

decision_summary["Share_%"] = (
    decision_summary["Store_Product_Records"]
    / len(inventory_base)
    * 100
).round(2)

display(decision_summary)

,Decision_Priority,Inventory_Decision,Store_Product_Records,Share_%
0,2,PRIORITY REPLENISHMENT,59,2.20
1,3,REPLENISH,51,1.90
2,4,MONITOR CUSTOMER ORDER,12,0.45
3,5,PROMOTION REVIEW,155,5.78
4,6,SLOW MOVER REVIEW,25,0.93
5,7,STOCK REVIEW,806,30.06
6,8,MONITOR,351,13.09
7,9,NO ACTION,1222,45.58


In [55]:
## Store-level decision profile
store_decision_summary = (
    inventory_base
    .pivot_table(
        index="Store",
        columns="Inventory_Decision",
        values="Product_ID",
        aggfunc="count",
        fill_value=0
    )
)

display(store_decision_summary)

Inventory_Decision,MONITOR,MONITOR CUSTOMER ORDER,NO ACTION,PRIORITY REPLENISHMENT,PROMOTION REVIEW,REPLENISH,SLOW MOVER REVIEW,STOCK REVIEW
Store,,,,,,,,
Belfast,33,0,172,6,2,6,0,164
Blanch,84,3,211,3,19,4,3,56
Cavan,101,0,182,2,36,5,7,50
Dundrum,5,0,198,14,2,10,0,154
Gorey,111,2,136,2,95,4,15,18
Navan,11,6,145,20,0,13,0,188
Sandyford,6,1,178,12,1,9,0,176


In [56]:
## Category-level decision profile
category_decision_summary = (
    inventory_base
    .groupby(
        ["Category", "Inventory_Decision"],
        as_index=False
    )
    .agg(
        Records=("Product_ID", "count"),
        Units_On_Hand=("Quantity", "sum")
    )
)

display(
    category_decision_summary.sort_values(
        ["Category", "Records"],
        ascending=[True, False]
    ).head(50)
)

,Category,Inventory_Decision,Records,Units_On_Hand
1,COOKER HOODS,NO ACTION,20,26
4,COOKER HOODS,STOCK REVIEW,13,0
0,COOKER HOODS,MONITOR,5,21
3,COOKER HOODS,REPLENISH,3,0
2,COOKER HOODS,PROMOTION REVIEW,1,71
6,COOKERS,NO ACTION,30,51
10,COOKERS,STOCK REVIEW,28,12
5,COOKERS,MONITOR,11,38
8,COOKERS,PROMOTION REVIEW,5,35
7,COOKERS,PRIORITY REPLENISHMENT,2,0


In [57]:
## Inspect highest-priority decisions
priority_inventory = (
    inventory_base[
        inventory_base["Inventory_Decision"] != "NO ACTION"
    ]
    .sort_values(
        [
            "Decision_Priority",
            "ABC_Priority",
            "Velocity_Priority",
            "Quantity"
        ],
        ascending=[True, False, False, True]
    )
)

display(
    priority_inventory[
        [
            "Product_ID",
            "Model",
            "Store",
            "Category",
            "Quantity",
            "ABC_Class",
            "Velocity_Band",
            "Stock_Band",
            "Inventory_Decision",
            "Decision_Priority",
            "Decision_Reason"
        ]
    ].head(30)
)

,Product_ID,Model,Store,Category,Quantity,ABC_Class,Velocity_Band,Stock_Band,Inventory_Decision,Decision_Priority,Decision_Reason
2472,1320,WEK365WCS,Blanch,WASHING MACHINES,0,A,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority
2476,1320,WEK365WCS,Navan,WASHING MACHINES,0,A,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority
2471,1320,WEK365WCS,Belfast,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT,2,Low stock with high sales velocity
2473,1320,WEK365WCS,Cavan,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT,2,Low stock with high sales velocity
2474,1320,WEK365WCS,Dundrum,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT,2,Low stock with high sales velocity
2477,1320,WEK365WCS,Sandyford,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT,2,Low stock with high sales velocity
405,439,EDHI618WD,Sandyford,TUMBLE DRYERS,0,A,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority
747,560,H7464BPBL,Navan,SINGLE OVENS,0,A,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority
2203,1213,TEH785WP,Navan,TUMBLE DRYERS,0,A,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority
93,255,AFB18432,Cavan,INT FREEZERS,0,A,LOW,OUT_OF_STOCK,PRIORITY REPLENISHMENT,2,Out of stock with high demand or A-class priority


In [58]:
## Unified decision validation
print("UNIFIED INVENTORY DECISION VALIDATION")
print("=" * 70)

print(
    "Total records:",
    len(inventory_base)
)

print(
    "Missing decisions:",
    inventory_base["Inventory_Decision"].isna().sum()
)

print(
    "Missing priorities:",
    inventory_base["Decision_Priority"].isna().sum()
)

print(
    "Missing reasons:",
    inventory_base["Decision_Reason"].isna().sum()
)

print(
    "Duplicate Product × Store:",
    inventory_base.duplicated(
        ["Product_ID", "Store"]
    ).sum()
)

print(
    "Decision count reconciliation:",
    decision_summary["Store_Product_Records"].sum()
    == len(inventory_base)
)

UNIFIED INVENTORY DECISION VALIDATION
Total records: 2681
Missing decisions: 0
Missing priorities: 0
Missing reasons: 0
Duplicate Product × Store: 0
Decision count reconciliation: True


The replenishment and slow-mover engines were successfully consolidated into
a single management-facing inventory decision layer at Product × Store grain.

The final decision distribution is:

- 59 → Priority Replenishment
- 51 → Replenish
- 12 → Monitor Customer Order
- 155 → Promotion Review
- 25 → Slow Mover Review
- 806 → Stock Review
- 351 → Monitor
- 1,222 → No Action

No records currently qualify for `URGENT REPLENISHMENT` because all negative
stock positions are associated with outstanding customer orders.

Validation confirmed:

- 2,681 total Product × Store records,
- zero missing inventory decisions,
- zero missing decision priorities,
- zero missing decision reasons,
- zero duplicate Product × Store records,
- full decision-count reconciliation.

Store- and category-level profiling also confirms that the unified decision
layer can support operational drill-down in downstream Power BI reporting.

The unified inventory decision layer therefore passes the current Phase 5
validation gate.

### Step 8 — Rule Boundary & Scenario Testing

In [59]:
# Create controlled scenarios
test_scenarios = pd.DataFrame([
    # 1
    {
        "Scenario": "A_HIGH_OUT_OF_STOCK",
        "Quantity": 0,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "HIGH",
        "ABC_Class": "A"
    },

    # 2
    {
        "Scenario": "B_MEDIUM_OUT_OF_STOCK",
        "Quantity": 0,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "MEDIUM",
        "ABC_Class": "B"
    },

    # 3
    {
        "Scenario": "A_HIGH_LOW_STOCK",
        "Quantity": 1,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "HIGH",
        "ABC_Class": "A"
    },

    # 4
    {
        "Scenario": "A_LOW_LOW_STOCK",
        "Quantity": 2,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "LOW",
        "ABC_Class": "A"
    },

    # 5
    {
        "Scenario": "C_LOW_HIGH_STOCK",
        "Quantity": 10,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "LOW",
        "ABC_Class": "C"
    },

    # 6
    {
        "Scenario": "NO_SALES_HIGH_STOCK",
        "Quantity": 20,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "NO_SALES",
        "ABC_Class": "UNCLASSIFIED"
    },

    # 7
    {
        "Scenario": "NO_SALES_ADEQUATE_STOCK",
        "Quantity": 4,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "NO_SALES",
        "ABC_Class": "UNCLASSIFIED"
    },

    # 8
    {
        "Scenario": "NEGATIVE_WITH_CUSTOMER_ORDER",
        "Quantity": -1,
        "Outstanding_Order_Qty": 1,
        "Velocity_Band": "MEDIUM",
        "ABC_Class": "A"
    },

    # 9
    {
        "Scenario": "NEGATIVE_WITHOUT_ORDER",
        "Quantity": -1,
        "Outstanding_Order_Qty": 0,
        "Velocity_Band": "HIGH",
        "ABC_Class": "A"
    }
])

display(test_scenarios)

,Scenario,Quantity,Outstanding_Order_Qty,Velocity_Band,ABC_Class
0,A_HIGH_OUT_OF_STOCK,0,0,HIGH,A
1,B_MEDIUM_OUT_OF_STOCK,0,0,MEDIUM,B
2,A_HIGH_LOW_STOCK,1,0,HIGH,A
3,A_LOW_LOW_STOCK,2,0,LOW,A
4,C_LOW_HIGH_STOCK,10,0,LOW,C
5,NO_SALES_HIGH_STOCK,20,0,NO_SALES,UNCLASSIFIED
6,NO_SALES_ADEQUATE_STOCK,4,0,NO_SALES,UNCLASSIFIED
7,NEGATIVE_WITH_CUSTOMER_ORDER,-1,1,MEDIUM,A
8,NEGATIVE_WITHOUT_ORDER,-1,0,HIGH,A


In [60]:
## Run all three engines
test_scenarios["Stock_Band"] = (
    test_scenarios["Quantity"]
    .apply(assign_stock_band)
)

test_scenarios["Reorder_Recommendation"] = (
    test_scenarios.apply(
        reorder_recommendation,
        axis=1
    )
)

test_scenarios["Slow_Mover_Action"] = (
    test_scenarios.apply(
        slow_mover_action,
        axis=1
    )
)

test_scenarios["Inventory_Decision"] = (
    test_scenarios.apply(
        assign_inventory_decision,
        axis=1
    )
)

display(
    test_scenarios[
        [
            "Scenario",
            "Quantity",
            "Velocity_Band",
            "ABC_Class",
            "Stock_Band",
            "Reorder_Recommendation",
            "Slow_Mover_Action",
            "Inventory_Decision"
        ]
    ]
)

,Scenario,Quantity,Velocity_Band,ABC_Class,Stock_Band,Reorder_Recommendation,Slow_Mover_Action,Inventory_Decision
0,A_HIGH_OUT_OF_STOCK,0,HIGH,A,OUT_OF_STOCK,REORDER - HIGH PRIORITY,NO ACTION,PRIORITY REPLENISHMENT
1,B_MEDIUM_OUT_OF_STOCK,0,MEDIUM,B,OUT_OF_STOCK,REORDER,NO ACTION,REPLENISH
2,A_HIGH_LOW_STOCK,1,HIGH,A,LOW,REORDER - HIGH PRIORITY,NO ACTION,PRIORITY REPLENISHMENT
3,A_LOW_LOW_STOCK,2,LOW,A,LOW,REVIEW,NO ACTION,STOCK REVIEW
4,C_LOW_HIGH_STOCK,10,LOW,C,HIGH,NO ACTION,SLOW MOVER REVIEW,SLOW MOVER REVIEW
5,NO_SALES_HIGH_STOCK,20,NO_SALES,UNCLASSIFIED,HIGH,NO ACTION,PROMOTION / MARKDOWN REVIEW,PROMOTION REVIEW
6,NO_SALES_ADEQUATE_STOCK,4,NO_SALES,UNCLASSIFIED,ADEQUATE,NO ACTION,MONITOR,MONITOR
7,NEGATIVE_WITH_CUSTOMER_ORDER,-1,MEDIUM,A,NEGATIVE,MONITOR CUSTOMER ORDER,NO ACTION,MONITOR CUSTOMER ORDER
8,NEGATIVE_WITHOUT_ORDER,-1,HIGH,A,NEGATIVE,REORDER - CRITICAL,NO ACTION,URGENT REPLENISHMENT


In [61]:
expected_results = {
    "A_HIGH_OUT_OF_STOCK":
        "PRIORITY REPLENISHMENT",

    "B_MEDIUM_OUT_OF_STOCK":
        "REPLENISH",

    "A_HIGH_LOW_STOCK":
        "PRIORITY REPLENISHMENT",

    "A_LOW_LOW_STOCK":
        "STOCK REVIEW",

    "C_LOW_HIGH_STOCK":
        "SLOW MOVER REVIEW",

    "NO_SALES_HIGH_STOCK":
        "PROMOTION REVIEW",

    "NO_SALES_ADEQUATE_STOCK":
        "MONITOR",

    "NEGATIVE_WITH_CUSTOMER_ORDER":
        "MONITOR CUSTOMER ORDER",

    "NEGATIVE_WITHOUT_ORDER":
        "URGENT REPLENISHMENT"
}

test_scenarios["Expected_Decision"] = (
    test_scenarios["Scenario"]
    .map(expected_results)
)

test_scenarios["Test_Passed"] = (
    test_scenarios["Inventory_Decision"]
    == test_scenarios["Expected_Decision"]
)

display(
    test_scenarios[
        [
            "Scenario",
            "Inventory_Decision",
            "Expected_Decision",
            "Test_Passed"
        ]
    ]
)

,Scenario,Inventory_Decision,Expected_Decision,Test_Passed
0,A_HIGH_OUT_OF_STOCK,PRIORITY REPLENISHMENT,PRIORITY REPLENISHMENT,True
1,B_MEDIUM_OUT_OF_STOCK,REPLENISH,REPLENISH,True
2,A_HIGH_LOW_STOCK,PRIORITY REPLENISHMENT,PRIORITY REPLENISHMENT,True
3,A_LOW_LOW_STOCK,STOCK REVIEW,STOCK REVIEW,True
4,C_LOW_HIGH_STOCK,SLOW MOVER REVIEW,SLOW MOVER REVIEW,True
5,NO_SALES_HIGH_STOCK,PROMOTION REVIEW,PROMOTION REVIEW,True
6,NO_SALES_ADEQUATE_STOCK,MONITOR,MONITOR,True
7,NEGATIVE_WITH_CUSTOMER_ORDER,MONITOR CUSTOMER ORDER,MONITOR CUSTOMER ORDER,True
8,NEGATIVE_WITHOUT_ORDER,URGENT REPLENISHMENT,URGENT REPLENISHMENT,True


In [62]:
passed_tests = test_scenarios["Test_Passed"].sum()
total_tests = len(test_scenarios)

print("INVENTORY RULE ENGINE TEST RESULTS")
print("=" * 70)

print(f"Tests passed: {passed_tests}/{total_tests}")
print(
    "All business-rule tests passed:",
    test_scenarios["Test_Passed"].all()
)

assert test_scenarios["Test_Passed"].all(), (
    "Inventory decision engine contains a failed business-rule scenario."
)

print("\n✓ Inventory decision rules validated successfully.")

INVENTORY RULE ENGINE TEST RESULTS
Tests passed: 9/9
All business-rule tests passed: True

✓ Inventory decision rules validated successfully.


### Step 9 — Inventory Segmentation

In [63]:
## Create management segments
def assign_inventory_segment(row):

    decision = row["Inventory_Decision"]

    if decision == "URGENT REPLENISHMENT":
        return "CRITICAL STOCK RISK"

    if decision == "PRIORITY REPLENISHMENT":
        return "HIGH PRIORITY REPLENISHMENT"

    if decision == "REPLENISH":
        return "REPLENISHMENT REQUIRED"

    if decision == "MONITOR CUSTOMER ORDER":
        return "CUSTOMER ORDER COMMITMENT"

    if decision == "PROMOTION REVIEW":
        return "EXCESS / NON-MOVING STOCK"

    if decision == "SLOW MOVER REVIEW":
        return "SLOW-MOVING STOCK"

    if decision == "STOCK REVIEW":
        return "STOCK AVAILABILITY REVIEW"

    if decision == "MONITOR":
        return "DEMAND MONITORING"

    return "HEALTHY / NO INTERVENTION"


inventory_base["Inventory_Segment"] = (
    inventory_base.apply(
        assign_inventory_segment,
        axis=1
    )
)

In [64]:
segment_summary = (
    inventory_base
    .groupby(
        ["Decision_Priority", "Inventory_Segment"],
        as_index=False
    )
    .agg(
        Store_Product_Records=("Product_ID", "count"),
        Units_On_Hand=("Quantity", "sum")
    )
    .sort_values("Decision_Priority")
)

segment_summary["Record_Share_%"] = (
    segment_summary["Store_Product_Records"]
    / len(inventory_base)
    * 100
).round(2)

display(segment_summary)

,Decision_Priority,Inventory_Segment,Store_Product_Records,Units_On_Hand,Record_Share_%
0,2,HIGH PRIORITY REPLENISHMENT,59,4,2.20
1,3,REPLENISHMENT REQUIRED,51,30,1.90
2,4,CUSTOMER ORDER COMMITMENT,12,-13,0.45
3,5,EXCESS / NON-MOVING STOCK,155,2439,5.78
4,6,SLOW-MOVING STOCK,25,249,0.93
5,7,STOCK AVAILABILITY REVIEW,806,121,30.06
6,8,DEMAND MONITORING,351,1282,13.09
7,9,HEALTHY / NO INTERVENTION,1222,1889,45.58


In [65]:
## ABC × Velocity management matrix
abc_velocity_matrix = (
    inventory_base[
        inventory_base["Demand_Status"] == "OBSERVED_SALES"
    ]
    .pivot_table(
        index="ABC_Class",
        columns="Velocity_Band",
        values="Product_ID",
        aggfunc="count",
        fill_value=0
    )
)

display(abc_velocity_matrix)

Velocity_Band,HIGH,LOW,MEDIUM
ABC_Class,,,
A,7,196,42
B,0,112,0
C,0,35,0


In [66]:
## Stock × velocity matrix
stock_velocity_matrix = (
    inventory_base
    .pivot_table(
        index="Stock_Band",
        columns="Velocity_Band",
        values="Product_ID",
        aggfunc="count",
        fill_value=0
    )
)

display(stock_velocity_matrix)

Velocity_Band,HIGH,LOW,MEDIUM,NO_SALES
Stock_Band,,,,
ADEQUATE,1,57,6,351
HIGH,0,25,10,155
LOW,4,181,23,1063
NEGATIVE,0,2,0,10
OUT_OF_STOCK,2,78,3,710


In [67]:
## Store inventory-health summary
store_inventory_health = (
    inventory_base
    .groupby("Store", as_index=False)
    .agg(
        Total_SKU_Records=("Product_ID", "count"),

        Units_On_Hand=("Quantity", "sum"),

        Out_Of_Stock=(
            "Stock_Band",
            lambda x: (x == "OUT_OF_STOCK").sum()
        ),

        Low_Stock=(
            "Stock_Band",
            lambda x: (x == "LOW").sum()
        ),

        High_Stock=(
            "Stock_Band",
            lambda x: (x == "HIGH").sum()
        ),

        Priority_Replenishment=(
            "Inventory_Decision",
            lambda x: (x == "PRIORITY REPLENISHMENT").sum()
        ),

        Replenishment_Required=(
            "Inventory_Decision",
            lambda x: (x == "REPLENISH").sum()
        ),

        Promotion_Review=(
            "Inventory_Decision",
            lambda x: (x == "PROMOTION REVIEW").sum()
        ),

        Slow_Mover_Review=(
            "Inventory_Decision",
            lambda x: (x == "SLOW MOVER REVIEW").sum()
        )
    )
)

display(store_inventory_health)

,Store,Total_SKU_Records,Units_On_Hand,Out_Of_Stock,Low_Stock,High_Stock,Priority_Replenishment,Replenishment_Required,Promotion_Review,Slow_Mover_Review
0,Belfast,383,405,152,190,2,6,6,2,0
1,Blanch,383,871,48,203,24,3,4,19,3
2,Cavan,383,1094,39,182,47,2,5,36,7
3,Dundrum,383,293,156,218,2,14,10,2,0
4,Gorey,383,2850,18,119,114,2,4,95,15
5,Navan,383,234,207,157,0,20,13,0,0
6,Sandyford,383,254,173,202,1,12,9,1,0


In [70]:
store_inventory_health["Out_Of_Stock_%"] = (
    store_inventory_health["Out_Of_Stock"]
    / store_inventory_health["Total_SKU_Records"]
    * 100
).round(2)

store_inventory_health["Low_Stock_%"] = (
    store_inventory_health["Low_Stock"]
    / store_inventory_health["Total_SKU_Records"]
    * 100
).round(2)

store_inventory_health["High_Stock_%"] = (
    store_inventory_health["High_Stock"]
    / store_inventory_health["Total_SKU_Records"]
    * 100
).round(2)

display(store_inventory_health)

,Store,Total_SKU_Records,Units_On_Hand,Out_Of_Stock,Low_Stock,High_Stock,Priority_Replenishment,Replenishment_Required,Promotion_Review,Slow_Mover_Review,Out_Of_Stock_%,Low_Stock_%,High_Stock_%
0,Belfast,383,405,152,190,2,6,6,2,0,39.69,49.61,0.52
1,Blanch,383,871,48,203,24,3,4,19,3,12.53,53.00,6.27
2,Cavan,383,1094,39,182,47,2,5,36,7,10.18,47.52,12.27
3,Dundrum,383,293,156,218,2,14,10,2,0,40.73,56.92,0.52
4,Gorey,383,2850,18,119,114,2,4,95,15,4.70,31.07,29.77
5,Navan,383,234,207,157,0,20,13,0,0,54.05,40.99,0.00
6,Sandyford,383,254,173,202,1,12,9,1,0,45.17,52.74,0.26


In [71]:
print("INVENTORY SEGMENTATION VALIDATION")
print("=" * 70)

print(
    "Total inventory records:",
    len(inventory_base)
)

print(
    "Missing inventory segments:",
    inventory_base["Inventory_Segment"].isna().sum()
)

print(
    "Segment reconciliation:",
    segment_summary["Store_Product_Records"].sum()
    == len(inventory_base)
)

print(
    "Duplicate Product × Store:",
    inventory_base.duplicated(
        ["Product_ID", "Store"]
    ).sum()
)

print(
    "Stores represented:",
    inventory_base["Store"].nunique()
)

print(
    "Inventory segments:",
    inventory_base["Inventory_Segment"].nunique()
)

INVENTORY SEGMENTATION VALIDATION
Total inventory records: 2681
Missing inventory segments: 0
Segment reconciliation: True
Duplicate Product × Store: 0
Stores represented: 7
Inventory segments: 8


### Step 10 — Inventory Opportunity & Risk Analysis

This section converts the validated inventory decision layer into
management-level business insights.

The analysis focuses on:

- stock-out and low-stock exposure,
- priority replenishment risk,
- A-class product availability,
- excess and non-moving inventory,
- slow-moving stock,
- store-level inventory risk,
- category-level inventory risk,
- inventory concentration and intervention opportunities.

These findings provide the analytical foundation for downstream Power BI
reporting and AI-assisted inventory recommendations.

In [72]:
## Overall inventory health KPIs
total_records = len(inventory_base)

inventory_kpis = pd.DataFrame({
    "KPI": [
        "Store-Product Records",
        "Units On Hand",
        "Out of Stock Records",
        "Low Stock Records",
        "High Stock Records",
        "Priority Replenishment",
        "Replenishment Required",
        "Promotion Review",
        "Slow Mover Review",
        "Customer Orders to Monitor"
    ],

    "Value": [
        total_records,
        inventory_base["Quantity"].sum(),

        (inventory_base["Stock_Band"] == "OUT_OF_STOCK").sum(),

        (inventory_base["Stock_Band"] == "LOW").sum(),

        (inventory_base["Stock_Band"] == "HIGH").sum(),

        (
            inventory_base["Inventory_Decision"]
            == "PRIORITY REPLENISHMENT"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "REPLENISH"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "PROMOTION REVIEW"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "SLOW MOVER REVIEW"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "MONITOR CUSTOMER ORDER"
        ).sum()
    ]
})

display(inventory_kpis)

,KPI,Value
0,Store-Product Records,2681
1,Units On Hand,6001
2,Out of Stock Records,793
3,Low Stock Records,1271
4,High Stock Records,190
5,Priority Replenishment,59
6,Replenishment Required,51
7,Promotion Review,155
8,Slow Mover Review,25
9,Customer Orders to Monitor,12


In [73]:
## A-class inventory risk
a_class_risk = inventory_base[
    (inventory_base["ABC_Class"] == "A") &
    (
        inventory_base["Stock_Band"]
        .isin(["NEGATIVE", "OUT_OF_STOCK", "LOW"])
    )
].copy()

print("A-CLASS INVENTORY RISK")
print("=" * 70)

print(
    "A-class store-product records at stock risk:",
    len(a_class_risk)
)

print("\nStock position:")
print(
    a_class_risk["Stock_Band"]
    .value_counts()
)

print("\nInventory decisions:")
print(
    a_class_risk["Inventory_Decision"]
    .value_counts()
)

A-CLASS INVENTORY RISK
A-class store-product records at stock risk: 179

Stock position:
Stock_Band
LOW             123
OUT_OF_STOCK     55
NEGATIVE          1
Name: count, dtype: int64

Inventory decisions:
Inventory_Decision
STOCK REVIEW              96
PRIORITY REPLENISHMENT    59
REPLENISH                 23
MONITOR CUSTOMER ORDER     1
Name: count, dtype: int64


In [74]:
display(
    a_class_risk.sort_values(
        ["Decision_Priority", "Velocity_Priority", "Quantity"],
        ascending=[True, False, True]
    )[
        [
            "Product_ID",
            "Model",
            "Store",
            "Category",
            "Quantity",
            "Velocity_Band",
            "Stock_Band",
            "Inventory_Decision",
            "Decision_Reason"
        ]
    ].head(30)
)

,Product_ID,Model,Store,Category,Quantity,Velocity_Band,Stock_Band,Inventory_Decision,Decision_Reason
2472,1320,WEK365WCS,Blanch,WASHING MACHINES,0,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority
2476,1320,WEK365WCS,Navan,WASHING MACHINES,0,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority
2471,1320,WEK365WCS,Belfast,WASHING MACHINES,1,HIGH,LOW,PRIORITY REPLENISHMENT,Low stock with high sales velocity
2473,1320,WEK365WCS,Cavan,WASHING MACHINES,1,HIGH,LOW,PRIORITY REPLENISHMENT,Low stock with high sales velocity
2474,1320,WEK365WCS,Dundrum,WASHING MACHINES,1,HIGH,LOW,PRIORITY REPLENISHMENT,Low stock with high sales velocity
2477,1320,WEK365WCS,Sandyford,WASHING MACHINES,1,HIGH,LOW,PRIORITY REPLENISHMENT,Low stock with high sales velocity
405,439,EDHI618WD,Sandyford,TUMBLE DRYERS,0,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority
747,560,H7464BPBL,Navan,SINGLE OVENS,0,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority
2203,1213,TEH785WP,Navan,TUMBLE DRYERS,0,MEDIUM,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority
93,255,AFB18432,Cavan,INT FREEZERS,0,LOW,OUT_OF_STOCK,PRIORITY REPLENISHMENT,Out of stock with high demand or A-class priority


In [75]:
## Store risk profile
store_risk_profile = store_inventory_health.copy()

store_risk_profile["Replenishment_Total"] = (
    store_risk_profile["Priority_Replenishment"]
    +
    store_risk_profile["Replenishment_Required"]
)

store_risk_profile["Excess_Stock_Actions"] = (
    store_risk_profile["Promotion_Review"]
    +
    store_risk_profile["Slow_Mover_Review"]
)

store_risk_profile = store_risk_profile.sort_values(
    [
        "Priority_Replenishment",
        "Replenishment_Total",
        "Out_Of_Stock"
    ],
    ascending=False
)

display(store_risk_profile)

,Store,Total_SKU_Records,Units_On_Hand,Out_Of_Stock,Low_Stock,High_Stock,Priority_Replenishment,Replenishment_Required,Promotion_Review,Slow_Mover_Review,Out_Of_Stock_%,Low_Stock_%,High_Stock_%,Replenishment_Total,Excess_Stock_Actions
5,Navan,383,234,207,157,0,20,13,0,0,54.05,40.99,0.00,33,0
3,Dundrum,383,293,156,218,2,14,10,2,0,40.73,56.92,0.52,24,2
6,Sandyford,383,254,173,202,1,12,9,1,0,45.17,52.74,0.26,21,1
0,Belfast,383,405,152,190,2,6,6,2,0,39.69,49.61,0.52,12,2
1,Blanch,383,871,48,203,24,3,4,19,3,12.53,53.00,6.27,7,22
2,Cavan,383,1094,39,182,47,2,5,36,7,10.18,47.52,12.27,7,43
4,Gorey,383,2850,18,119,114,2,4,95,15,4.70,31.07,29.77,6,110


In [76]:
## Category replenishment risk
category_replenishment_risk = (
    inventory_base[
        inventory_base["Inventory_Decision"].isin([
            "PRIORITY REPLENISHMENT",
            "REPLENISH"
        ])
    ]
    .groupby("Category", as_index=False)
    .agg(
        Replenishment_Records=("Product_ID", "count"),
        Current_Units=("Quantity", "sum")
    )
    .sort_values(
        "Replenishment_Records",
        ascending=False
    )
)

display(category_replenishment_risk.head(15))

,Category,Replenishment_Records,Current_Units
13,SINGLE OVENS,26,11
16,WASHING MACHINES,14,10
8,INT FREEZERS,13,0
14,TUMBLE DRYERS,13,10
7,INT DISHWASHERS,7,3
6,HOBS,6,0
12,MICROWAVE OVENS,6,0
11,INT MICROWAVES,5,0
10,INT FRIDGES,4,0
0,COOKER HOODS,3,0


In [77]:
## Excess-stock exposure
excess_stock = inventory_base[
    inventory_base["Inventory_Decision"].isin([
        "PROMOTION REVIEW",
        "SLOW MOVER REVIEW"
    ])
].copy()

excess_summary = (
    excess_stock
    .groupby("Category", as_index=False)
    .agg(
        Actionable_Records=("Product_ID", "count"),
        Units_On_Hand=("Quantity", "sum")
    )
    .sort_values(
        "Units_On_Hand",
        ascending=False
    )
)

print(
    "Total excess/slow-moving records:",
    len(excess_stock)
)

print(
    "Units currently represented by these records:",
    excess_stock["Quantity"].sum()
)

display(excess_summary.head(15))

Total excess/slow-moving records: 180
Units currently represented by these records: 2688


,Category,Actionable_Records,Units_On_Hand
16,SINGLE OVENS,25,371
21,WASHING MACHINES,14,351
14,MICROWAVE OVENS,13,310
17,TUMBLE DRYERS,20,281
8,HOBS,25,278
9,INT DISHWASHERS,16,266
11,INT FRIDGE FREEZERS,10,222
0,COOKER HOODS,1,71
13,INT MICROWAVES,6,71
2,DOUBLE OVENS,7,70


In [78]:
## Store excess-stock exposure
store_excess_summary = (
    excess_stock
    .groupby("Store", as_index=False)
    .agg(
        Actionable_Records=("Product_ID", "count"),
        Units_On_Hand=("Quantity", "sum")
    )
    .sort_values(
        "Units_On_Hand",
        ascending=False
    )
)

display(store_excess_summary)

,Store,Actionable_Records,Units_On_Hand
4,Gorey,110,2126
2,Cavan,43,367
1,Blanch,22,164
3,Dundrum,2,13
0,Belfast,2,12
5,Sandyford,1,6


In [79]:
## High-value + high-velocity availability risk
critical_commercial_risk = inventory_base[
    (inventory_base["ABC_Class"] == "A") &
    (inventory_base["Velocity_Band"] == "HIGH") &
    (
        inventory_base["Stock_Band"]
        .isin(["OUT_OF_STOCK", "LOW"])
    )
].copy()

print("HIGH COMMERCIAL AVAILABILITY RISK")
print("=" * 70)

print(
    "A-class + High Velocity + OOS/Low Stock records:",
    len(critical_commercial_risk)
)

display(
    critical_commercial_risk[
        [
            "Product_ID",
            "Model",
            "Store",
            "Category",
            "Quantity",
            "ABC_Class",
            "Velocity_Band",
            "Stock_Band",
            "Inventory_Decision"
        ]
    ].sort_values(
        ["Quantity", "Store"]
    )
)

HIGH COMMERCIAL AVAILABILITY RISK
A-class + High Velocity + OOS/Low Stock records: 6


,Product_ID,Model,Store,Category,Quantity,ABC_Class,Velocity_Band,Stock_Band,Inventory_Decision
2472,1320,WEK365WCS,Blanch,WASHING MACHINES,0,A,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT
2476,1320,WEK365WCS,Navan,WASHING MACHINES,0,A,HIGH,OUT_OF_STOCK,PRIORITY REPLENISHMENT
2471,1320,WEK365WCS,Belfast,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT
2473,1320,WEK365WCS,Cavan,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT
2474,1320,WEK365WCS,Dundrum,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT
2477,1320,WEK365WCS,Sandyford,WASHING MACHINES,1,A,HIGH,LOW,PRIORITY REPLENISHMENT


In [80]:
## Create an intervention summary
intervention_summary = pd.DataFrame({

    "Intervention": [
        "Priority Replenishment",
        "Standard Replenishment",
        "Promotion / Markdown Review",
        "Slow-Mover Review",
        "Customer Order Monitoring"
    ],

    "Records": [
        (
            inventory_base["Inventory_Decision"]
            == "PRIORITY REPLENISHMENT"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "REPLENISH"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "PROMOTION REVIEW"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "SLOW MOVER REVIEW"
        ).sum(),

        (
            inventory_base["Inventory_Decision"]
            == "MONITOR CUSTOMER ORDER"
        ).sum()
    ]
})

intervention_summary["Share_%"] = (
    intervention_summary["Records"]
    / len(inventory_base)
    * 100
).round(2)

display(intervention_summary)

,Intervention,Records,Share_%
0,Priority Replenishment,59,2.20
1,Standard Replenishment,51,1.90
2,Promotion / Markdown Review,155,5.78
3,Slow-Mover Review,25,0.93
4,Customer Order Monitoring,12,0.45


#### Key Business Findings

The inventory intelligence analysis identified a significant imbalance between
product availability risk and excess stock exposure across the retail network.

- The inventory layer contains 2,681 unique store-product records representing
  6,001 units currently on hand.

- 793 store-product records are out of stock and 1,271 are classified as
  low stock. However, the decision engine prevents blanket replenishment by
  identifying only 110 records requiring replenishment action:
  59 Priority Replenishment and 51 Standard Replenishment.

- 179 A-class store-product records are exposed to stock risk, comprising
  123 low-stock records, 55 out-of-stock records and 1 negative-stock record.
  This highlights availability exposure among commercially important products.

- Six records represent particularly high commercial availability risk because
  they combine A-class importance, high sales velocity and either low or
  zero stock. These records relate to the WEK365WCS washing-machine model
  across multiple stores.

- Replenishment exposure is highest in Single Ovens (26 records), followed by
  Washing Machines (14), Integrated Freezers (13) and Tumble Dryers (13).

- 180 store-product records were identified for excess-stock or slow-mover
  intervention, representing 2,688 units of inventory.

- Gorey represents the largest excess-stock concentration, with 110 actionable
  records containing 2,126 units. This represents approximately 79.1% of all
  units currently associated with promotion or slow-mover interventions.

- The store-level results indicate a material inventory allocation imbalance.
  For example, Navan has 207 out-of-stock records with only 234 units on hand,
  while Gorey has 2,850 units on hand and only 18 out-of-stock records.

- Some categories simultaneously exhibit replenishment requirements and
  excess-stock exposure across the network. This creates an opportunity for
  future inter-store transfer optimisation before external replenishment is
  triggered.

Overall, the analysis demonstrates that the inventory challenge is not simply
a shortage of stock. The evidence indicates a combination of availability
risk, excess-stock concentration and potential inventory allocation
inefficiency across stores.

The decision layer therefore provides a governed foundation for replenishment,
stock review, promotion/markdown review, slow-mover management and future
inter-store inventory optimisation.

### Step 11 - Final Governed Phase 5 Output Tables.

In [84]:
# ============================================================
#  BUILD GOVERNED PHASE 5 OUTPUT TABLES
# ============================================================

# ------------------------------------------------------------
# 1. Main inventory decision layer
# ------------------------------------------------------------

inventory_decision_layer = inventory_base.copy()


# ------------------------------------------------------------
# 2. Inventory segment summary
# ------------------------------------------------------------

inventory_segment_summary = (
    inventory_decision_layer
    .groupby("Inventory_Segment", dropna=False)
    .agg(
        Store_Product_Records=("Product_ID", "size"),
        Units_On_Hand=("Quantity", "sum")
    )
    .reset_index()
)

inventory_segment_summary["Share_%"] = (
    inventory_segment_summary["Store_Product_Records"]
    / len(inventory_decision_layer)
    * 100
).round(2)


# ------------------------------------------------------------
# 3. Store inventory health
# ------------------------------------------------------------

store_inventory_health = (
    inventory_decision_layer
    .groupby("Store")
    .agg(
        Total_SKU_Records=("Product_ID", "size"),
        Units_On_Hand=("Quantity", "sum"),

        Out_Of_Stock=(
            "Stock_Band",
            lambda x: (x == "OUT_OF_STOCK").sum()
        ),

        Low_Stock=(
            "Stock_Band",
            lambda x: (x == "LOW").sum()
        ),

        High_Stock=(
            "Stock_Band",
            lambda x: (x == "HIGH").sum()
        ),

        Priority_Replenishment=(
            "Inventory_Decision",
            lambda x: (x == "PRIORITY REPLENISHMENT").sum()
        ),

        Replenishment_Required=(
            "Inventory_Decision",
            lambda x: (x == "REPLENISH").sum()
        ),

        Promotion_Review=(
            "Inventory_Decision",
            lambda x: (x == "PROMOTION REVIEW").sum()
        ),

        Slow_Mover_Review=(
            "Inventory_Decision",
            lambda x: (x == "SLOW MOVER REVIEW").sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Category inventory health
# ------------------------------------------------------------

category_inventory_health = (
    inventory_decision_layer
    .groupby("Category")
    .agg(
        Total_SKU_Records=("Product_ID", "size"),
        Units_On_Hand=("Quantity", "sum"),

        Out_Of_Stock=(
            "Stock_Band",
            lambda x: (x == "OUT_OF_STOCK").sum()
        ),

        Low_Stock=(
            "Stock_Band",
            lambda x: (x == "LOW").sum()
        ),

        High_Stock=(
            "Stock_Band",
            lambda x: (x == "HIGH").sum()
        ),

        Priority_Replenishment=(
            "Inventory_Decision",
            lambda x: (x == "PRIORITY REPLENISHMENT").sum()
        ),

        Replenishment_Required=(
            "Inventory_Decision",
            lambda x: (x == "REPLENISH").sum()
        ),

        Promotion_Review=(
            "Inventory_Decision",
            lambda x: (x == "PROMOTION REVIEW").sum()
        ),

        Slow_Mover_Review=(
            "Inventory_Decision",
            lambda x: (x == "SLOW MOVER REVIEW").sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Replenishment candidates
# ------------------------------------------------------------

replenishment_candidates = (
    inventory_decision_layer[
        inventory_decision_layer["Inventory_Decision"].isin(
            ["PRIORITY REPLENISHMENT", "REPLENISH"]
        )
    ]
    .copy()
    .sort_values(
        ["Decision_Priority", "ABC_Class", "Avg_Positive_Velocity"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Excess / slow-moving stock candidates
# ------------------------------------------------------------

excess_stock_candidates = (
    inventory_decision_layer[
        inventory_decision_layer["Inventory_Decision"].isin(
            ["PROMOTION REVIEW", "SLOW MOVER REVIEW"]
        )
    ]
    .copy()
    .sort_values(
        ["Quantity", "Decision_Priority"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. High commercial availability risk
# ------------------------------------------------------------

commercial_risk_candidates = (
    inventory_decision_layer[
        (inventory_decision_layer["ABC_Class"] == "A")
        & (inventory_decision_layer["Velocity_Band"] == "HIGH")
        & (inventory_decision_layer["Stock_Band"].isin(
            ["OUT_OF_STOCK", "LOW"]
        ))
    ]
    .copy()
    .sort_values(
        ["Quantity", "Avg_Positive_Velocity"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Intervention summary
# ------------------------------------------------------------

intervention_summary = (
    inventory_decision_layer
    .groupby(
        ["Decision_Priority", "Inventory_Decision"],
        dropna=False
    )
    .agg(
        Store_Product_Records=("Product_ID", "size"),
        Units_On_Hand=("Quantity", "sum")
    )
    .reset_index()
    .sort_values("Decision_Priority")
)

intervention_summary["Share_%"] = (
    intervention_summary["Store_Product_Records"]
    / len(inventory_decision_layer)
    * 100
).round(2)


print("Governed Phase 5 output tables created successfully.")

Governed Phase 5 output tables created successfully.


In [85]:
# ============================================================
#  OUTPUT TABLE REGISTRY
# ============================================================

phase5_outputs = {
    "inventory_decision_layer": inventory_decision_layer,
    "inventory_segment_summary": inventory_segment_summary,
    "store_inventory_health": store_inventory_health,
    "category_inventory_health": category_inventory_health,
    "replenishment_candidates": replenishment_candidates,
    "excess_stock_candidates": excess_stock_candidates,
    "commercial_risk_candidates": commercial_risk_candidates,
    "intervention_summary": intervention_summary
}


print("PHASE 5 GOVERNED OUTPUT REGISTRY")
print("=" * 65)

for name, df in phase5_outputs.items():
    print(
        f"{name:<32} "
        f"Rows: {df.shape[0]:>5} | "
        f"Columns: {df.shape[1]:>3}"
    )

PHASE 5 GOVERNED OUTPUT REGISTRY
inventory_decision_layer         Rows:  2681 | Columns:  27
inventory_segment_summary        Rows:     8 | Columns:   4
store_inventory_health           Rows:     7 | Columns:  10
category_inventory_health        Rows:    27 | Columns:  10
replenishment_candidates         Rows:   110 | Columns:  27
excess_stock_candidates          Rows:   180 | Columns:  27
commercial_risk_candidates       Rows:     6 | Columns:  27
intervention_summary             Rows:     8 | Columns:   5


In [86]:
# ============================================================
#  GOVERNED OUTPUT VALIDATION
# ============================================================

print("PHASE 5 OUTPUT VALIDATION")
print("=" * 65)

# Core grain validation
assert len(inventory_decision_layer) == 2681

assert (
    inventory_decision_layer[
        ["Product_ID", "Store"]
    ]
    .duplicated()
    .sum()
    == 0
)

# Decision completeness
assert inventory_decision_layer["Inventory_Decision"].isna().sum() == 0
assert inventory_decision_layer["Decision_Priority"].isna().sum() == 0
assert inventory_decision_layer["Decision_Reason"].isna().sum() == 0
assert inventory_decision_layer["Inventory_Segment"].isna().sum() == 0

# Candidate reconciliation
assert len(replenishment_candidates) == (
    inventory_decision_layer["Inventory_Decision"]
    .isin(["PRIORITY REPLENISHMENT", "REPLENISH"])
    .sum()
)

assert len(excess_stock_candidates) == (
    inventory_decision_layer["Inventory_Decision"]
    .isin(["PROMOTION REVIEW", "SLOW MOVER REVIEW"])
    .sum()
)

# Segment reconciliation
assert (
    inventory_segment_summary["Store_Product_Records"].sum()
    == len(inventory_decision_layer)
)

# Store reconciliation
assert (
    store_inventory_health["Total_SKU_Records"].sum()
    == len(inventory_decision_layer)
)

# Category reconciliation
assert (
    category_inventory_health["Total_SKU_Records"].sum()
    == len(inventory_decision_layer)
)

print("✓ Inventory grain validated")
print("✓ Product × Store uniqueness validated")
print("✓ Decision completeness validated")
print("✓ Inventory segmentation validated")
print("✓ Replenishment candidate reconciliation validated")
print("✓ Excess-stock candidate reconciliation validated")
print("✓ Store summary reconciliation validated")
print("✓ Category summary reconciliation validated")

print("\nAll governed Phase 5 outputs passed validation.")

PHASE 5 OUTPUT VALIDATION
✓ Inventory grain validated
✓ Product × Store uniqueness validated
✓ Decision completeness validated
✓ Inventory segmentation validated
✓ Replenishment candidate reconciliation validated
✓ Excess-stock candidate reconciliation validated
✓ Store summary reconciliation validated
✓ Category summary reconciliation validated

All governed Phase 5 outputs passed validation.


In [87]:
## Save the files
PHASE5_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phase5_inventory_intelligence"
)

PHASE5_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Phase 5 output directory:")
print(PHASE5_OUTPUT_DIR)

Phase 5 output directory:
d:\STUDY\Data_Science_Courses\PROJECTS\12.Smart_AI-Retail_System\Smart_AI_Retail_System\data\processed\phase5_inventory_intelligence


In [88]:
""" 
export_map = {
    "inventory_decision_layer.csv":
        inventory_decision_layer,

    "inventory_segment_summary.csv":
        inventory_segment_summary,

    "store_inventory_health.csv":
        store_inventory_health,

    "category_inventory_health.csv":
        category_inventory_health,

    "replenishment_candidates.csv":
        replenishment_candidates,

    "excess_stock_candidates.csv":
        excess_stock_candidates,

    "commercial_risk_candidates.csv":
        commercial_risk_candidates,

    "intervention_summary.csv":
        intervention_summary
}


for filename, df in export_map.items():

    output_path = PHASE5_OUTPUT_DIR / filename

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"✓ {filename:<40} "
        f"Rows: {len(df):>5}"
    ) """

✓ inventory_decision_layer.csv             Rows:  2681
✓ inventory_segment_summary.csv            Rows:     8
✓ store_inventory_health.csv               Rows:     7
✓ category_inventory_health.csv            Rows:    27
✓ replenishment_candidates.csv             Rows:   110
✓ excess_stock_candidates.csv              Rows:   180
✓ commercial_risk_candidates.csv           Rows:     6
✓ intervention_summary.csv                 Rows:     8
